> Итог этапа 06: решения приняты. Актуальная выборка `06_05_cases_excluded.parquet` содержит 6 331 485 строк и 18 349 GoA=True. Расчеты ниже сохраняют исторический вход этого шага. Общий итог: `europe/reports/attempts/README.md`.


# 06_02 – ВПП и источники для короткого интервала LSGG

Случай TAP948_37820: две отметки через 12 секунд, обозначения 22R и 22. Проверяем распределение таких обозначений во всей текущей выборке LSGG и наличие идентификатора в опубликованных материалах старого архива.

Источники: официальные отчеты аэропорта за 2017–2018 и EUROCONTROL по данным 2019 года. Полные PDF аэропорта сейчас недоступны; просмотренные индексированные фрагменты явно отделены от полной проверки документов. NOTAM для 01.03.2019 не получен, исторический AIP нужной даты не проверен. Отсутствие найденного NOTAM не доказывает отсутствие временных ограничений.

Гипотеза – неподходящий справочник ВПП или перекрывающиеся сегменты одного захода. Не исправляем 22R, не объединяем попытки и не меняем GoA по предположению.

In [1]:
# Находим проект и сохраняем разбор случая без исправления базы
from pathlib import Path
import sys
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'europe/scripts/review_lsgg_runway_case.py').exists())
sys.path.insert(0, str(ROOT / 'europe/scripts'))
import review_lsgg_runway_case as review
case, labels, patterns, journal = review.main()
display(case)
display(labels)
display(patterns)
print(journal)

,flight_id,icao24,airport,landing_time,attempt_times,ILS,n_attempts,GoA
0,TAP948_37820,495295,LSGG,2019-03-01 16:29:15+00:00,"[2019-03-01T16:29:03.000000, 2019-03-01T16:29:...","[22R, 22]",2,True


,runway_label,GoA,observations,attempts
0,04,False,80572,80572
1,04,True,195,367
2,04L,False,72,72
3,22R,False,70154,70154
4,22,False,54046,54046
5,22R,True,247,292
6,22,True,297,402
7,04L,True,1,1


,runway_sequence,GoA,observations
0,04,False,80572
3,22R,False,70154
4,22,False,54046
1,04|04,True,170
5,22R|22,True,167
8,22|22,True,105
2,04L,False,72
6,22R|22R,True,45
7,22|22R,True,18
9,22R|04,True,15


{'input_sha256': '1918930dbbf8c410f36524a0f4c5c4e2d98c01ccafb1b2bcb2d579c1d6d70f72', 'input_unchanged': True, 'airport': 'LSGG', 'airport_rows': 205374, 'airport_go_arounds': 530, 'case_flight_id': 'TAP948_37820', 'case_gap_seconds': 12, 'case_runway_labels': ['22R', '22'], 'case_GoA': True, 'archived_materials': [{'member': 'validation_sample/LSGG_04.html', 'sha256': '9ac0eac41945c8675ba506894d83ef9f94599ac47be814385e921345e48ff27b', 'target_identifier_found': False}, {'member': 'validation_sample/LSGG_22.html', 'sha256': 'ba0a034f0f573e1f52604fab4639c33cfee65bba79e7c1207c3c1689d0affc35', 'target_identifier_found': False}, {'member': 'validation_sample/LSGG_22R.html', 'sha256': '1b8af8cdee74b7340420662a451322ce57cd1be6a4b5d012ee6d3894a86e9fa8', 'target_identifier_found': False}], 'historical_notam_obtained': False, 'notam_target_utc': '2019-03-01 16:29:03 to 16:29:15', 'notam_search_result': 'No primary historical NOTAM for target interval retrieved; absence of notice not established'

## Результат

22R встречается в 70 401 наблюдении LSGG, включая 247 GoA=True. Это вопрос происхождения справочника ВПП, а не единичная опечатка, подтвержденная текущей проверкой. Исторические материалы поддерживают гипотезу, но полный AIP и NOTAM целевой даты не получены. Простая замена обозначения не подтверждает правильность числа попыток или метки. Данные сохранены. [Подробные свидетельства и ограничения](../../../../reports/attempts/06_02_review.md).

## Проверка кода разметки – 09.10.2026


Проверка 09.10.2026. Исходные данные и метки не изменены.

## Что доступно у авторов

Проверен [репозиторий OSN23-GoA](https://github.com/figuetbe/OSN23-GoA/tree/f5aaaefc6ebeb2cff8048d9fa7b1f1da2d73851d), commit f5aaaefc6ebeb2cff8048d9fa7b1f1da2d73851d. Ноутбук в ячейке 3 (индекс с нуля) читает готовую таблицу osn23_landings_merged.parquet.gzip. В текущем дереве нет отдельного кода первоначального формирования ILS, attempt_times и GoA. Это ограничение текущего опубликованного дерева, не утверждение о всех версиях или неопубликованном коде.

pyproject.toml задает traffic ^2.9.0, poetry.lock закрепляет 2.9.0. Это версия окружения анализа; ее использование при создании всей таблицы не доказано. Сохранены ноутбук и файлы окружения фиксированного commit, а также navigation.py и runways.py из wheel traffic 2.9.0. SHA256 wheel проверен по PyPI и совпадает с poetry.lock.

## Поля и порядок обработки

В [статье, раздел 2.1 и таблица 2](https://journals.open.tudelft.nl/joas/article/view/7219) ILS описано как список обозначений ВПП попыток; attempt_times – времена попыток; GoA – признак хотя бы одного ухода. Метод статьи требует двух сегментов, согласованных с ВПП, и фазы набора высоты между ними. Точная инструкция выбора времени каждой попытки из сегмента в опубликованном коде формирования не найдена.

Ячейка 21 берет первые два готовых attempt_times и вырезает траекторию между ними. Она использует разметку, а не создает ее. Ячейка 29 фильтрует рассчитанные участки для анализа времени и расстояния: ga_time_s > 200, d_start_end < 2, speed < 300. Этот фильтр не переписывает исходный GoA и не является правилом удаления строк из нашего набора. Ячейка 43 повторно вызывает aligned_on_ils с angle_tolerance=0.15 для оценки топлива A320; этот параметр нельзя автоматически считать параметром первоначальной разметки.

## Справочник и двойное определение

В traffic 2.9.0 runways.py загружает https://ourairports.com/data/runways.csv и кеширует пороги. download_runways не проверяет closed или surface: для каждой строки с допустимыми широтами формируются два порога. По дате полета справочник не выбирается. Следовательно, закрытая полоса может участвовать в геометрическом определении, если она есть в полученном файле. Исходный кеш авторов отсутствует; наличие именно 22R и 04L в нем пока не воспроизведено.

aligned_on_ils проверяет каждый порог отдельно. Геометрическое условие использует расстояние, умноженное на абсолютную разность направлений в радианах, и согласованность bearing с track. Это не просто допуск в градусах. Кандидаты разделяются по разрывам 20 секунд, требуют длительности более одной минуты при стандартном параметре и altitude_min < 5000. Все результаты сортируются по началу; удаления перекрытий между разными порогами нет. Высота здесь не является проверкой высоты над ВПП.

Выполнена минимальная проверка сохраненного тела функции: две искусственные одинаковые оси и один непрерывный снижающийся сегмент дают два полностью перекрывающихся результата с разными именами. Это демонстрация отсутствия устранения дублей, а не восстановление геометрии LSGG. На том же примере go_around не возвращает уход: CLIMB отсутствует. Адаптер подставляет искусственные bearing и distance; реальные измерения, геодезия и классификатор фаз не проверяются.

Таким образом, двойное геометрическое определение возможно, но утверждать, что оно породило GoA=True в нашем случае, пока нельзя. go_around использует дополнительную проверку CLIMB после начала первого сегмента и ищет следующий сегмент после окончания выбранной фазы набора. Начала и окончания реальных сегментов и результаты фаз для TAP948_37820 неизвестны.

## Решение по TAP948_37820

Две отметки 01.03.2019 16:29:03 и 16:29:15 UTC, ILS=[22R,22], n_attempts=2, GoA=True сохраняются. Смена обозначения и интервал 12 секунд – признаки для расследования; они не доказывают правильную замену GoA=False. Массовая нормализация 22R → 22 тоже не исправляет механизм счета попыток.

Следующий необходимый материал – код первоначального формирования таблицы либо исходная траектория с границами сегментов и фазами. До его получения случай остается неразрешенным. Остальные 184 перехода 22/22R, имеющие интервалы от 522 секунд, нельзя считать дублями только из-за обозначений. Весь LSGG не исключается.

Машинный журнал: 06_02_label_provenance.json. Воспроизведение из корня проекта: `.\.venv\Scripts\python.exe europe/scripts/audit_lsgg_label_provenance.py`. Скрипт не открывает и не меняет исходный Parquet.


## Решение об исключении – 09.10.2026

По прямому решению пользователя из дальнейшего эксперимента исключена одна запись: TAP948_37820, ICAO24 495295, LSGG, посадка 01.03.2019 16:29:15 UTC. Причина – неразрешенная неоднозначность разметки и прекращение дальнейшей проверки случая; это не подтверждение ошибочности GoA=True.

Актуальная выборка: `europe/data/analytical/06_02_lsgg_case_excluded.parquet` – 6,331,493 строк, 18,357 GoA=True. Для LSGG осталось 205,373 строк и 529 GoA=True. Полная исключенная строка сохранена в `europe/data/review/06_02_lsgg_excluded_record.parquet`. Вход этапа 05 сохранен; порядок, схема и значения всех остальных строк проверены после записи.

Исторические отчеты 06_01–06_03 описывают вход этапа 05 до исключения. Журнал решения: `europe/reports/attempts/LSGG/06_02_exclusion_applied.json`. Воспроизведение из корня: `.\.venv\Scripts\python.exe europe/scripts/exclude_lsgg_ambiguous_case.py`.


## Итог табличного разбора всего LSGG – 09.10.2026


Проверка 09.10.2026 выполнена на новой копии после исключения TAP948_37820: `europe/data/analytical/06_02_lsgg_case_excluded.parquet`. Проверены все 205 373 строки LSGG, включая 529 GoA=True. SHA256 входа соответствует журналу исключения и не изменился. Новые исправления и исключения не выполнялись.

## Распределение обозначений

| Обозначение в ILS | Строки с обозначением | GoA=True |
| --- | ---: | ---: |
| 04 | 80 767 | 195 |
| 04L | 73 | 1 |
| 22 | 54 342 | 296 |
| 22R | 70 400 | 246 |

Каждая строка учитывается один раз на обозначение; строки с несколькими обозначениями входят в несколько счетчиков. Сумма столбца не равна количеству уникальных записей.

22 и 22R встречаются одновременно во всех 53 месяцах и в 1 190 UTC-днях. Оба обозначения встречаются у 3 606 из 6 729 ICAO24-адресов и у 164 значений typecode. Адрес и код не подтверждают историческую идентичность конкретного самолета. Доля 22R среди строк с 22 или 22R по месяцам находится в пределах 34,76–75,03%; знаменатель здесь включает отдельные счетчики обозначений, а не уникальные полеты направления 22. Это противоречит простой версии единовременного переименования как объяснению смешения, но не доказывает конкретную причину.

44 из 73 строк с 04L имеют typecode B752, 11 – A320. Единственная положительная запись с 04L: LGL8367_29804, ICAO24 4d00ca, typecode DH8D, 03.02.2023, последовательность 04L → 04, отметки 19:45:21 и 19:57:33 UTC. Интервал 732 секунды. Это не второй аналог 12-секундного случая.

## Справочник и геометрия

Сохранены текущие строки [OurAirports runways.csv](https://ourairports.com/data/runways.csv), реестр URL и SHA256. Основная полоса обозначена 04/22, покрытие CON, closed=0; другая – 04L/22R, GRS, closed=1. В исследованном загрузчике traffic 2.9.0 closed и surface не используются для фильтрации. Кеш справочника авторов и дата его загрузки неизвестны. Полученный сегодня файл не доказывает состав справочника или статус аэродрома в конкретную дату 2019–2023 годов. Попытка найти исторический снимок через GitHub API вернула HTTP 403.

По текущим координатам сферическая оценка разницы направлений осей – около 0,90 градуса, расстояния между порогами 22 и 22R – 1 917 м. Оценка поперечного смещения порога 22R от основной оси – около 236 м. Это близкие, но не одинаковые оси; искусственный пример одинаковых осей из предыдущего разбора не воспроизводит LSGG. Координаты, реальные траектории и историческая версия справочника не были совместно воспроизведены.

## Число попыток и GoA

Проверен 531 соседний интервал; минимальный – 364 секунды. В 184 интервалах меняется 22/22R, еще в одном – 04L/04. Среди этих 185 интервалов минимальный составляет 522 секунды. Нарушений размеров списков, порядка времен, повторных времен и последней попытки после landing_time не обнаружено. GoA согласован с n_attempts > 1, что является внутренней согласованностью, а не доказательством реального ухода.

Условное сопоставление 22R → 22, 04L → 04 уменьшило бы количество уникальных обозначений в 185 записях, но не изменило бы число элементов attempt_times, n_attempts или GoA. Это диагностический сценарий; исходные ILS не заменялись. Слияние соседних попыток по одинаковому условному направлению уничтожило бы события, разделенные минутами, без доказательств дублирования.

Даже после этого условного сопоставления последнее обозначение расходится с AP_C_RWY в 772 строках, включая 5 GoA=True. Поля могут отражать разные правила и моменты присоединения; AP_C_RWY не используется как независимая истина и не определяет автоматическое исправление.

## Чувствительность состава выборки

| Группа | Строки | GoA=True | Метки GoA на 1 000 строк |
| --- | ---: | ---: | ---: |
| Все оставшиеся LSGG | 205 373 | 529 | 2,576 |
| Содержат 22R или 04L | 70 473 | 247 | 3,505 |
| Содержат только 04/22 | 134 900 | 282 | 2,090 |

Удаление строк с 22R/04L затронуло бы 34,3% наблюдений и 46,7% положительных меток LSGG. Это заметно меняет состав выборки; разница долей сама по себе не доказывает ошибочную разметку и не оценивает причинный эффект справочника. Проверка чувствительности моделей еще не проводилась: здесь сравниваются только состав и исходные метки.

## Решение по каждому вопросу

| Вопрос | Результат и действие |
| --- | --- |
| TAP948_37820, 12 секунд | Исключение уже применено по решению пользователя; строка сохранена отдельно |
| Однократное переименование объясняет 22/22R | Не объясняет совместное присутствие во всех месяцах |
| Закрытая полоса может попасть в детектор | Механизм подтвержден кодом загрузчика; текущий справочник содержит такую строку |
| 22R и 04L можно массово исправить | Исторический кеш и геометрическое присоединение не восстановлены; исправление не применяется |
| Две разные метки автоматически означают ложный GoA | Нет; алгоритм ухода требует CLIMB, а оставшиеся интервалы разделены минутами |
| Остальные строки полностью валидированы | Нет; подтверждена табличная согласованность, независимой траекторной проверки нет |
| Использовать ILS, attempt_times, n_attempts в раннем прогнозе | Не использовать: это сведения завершенного наблюдения |
| Использовать AP_C_RWY в раннем прогнозе | Доступность и смысл не установлены; ранее установленное ограничение сохраняется |

Табличный аудит завершен. Для научно обоснованной массовой коррекции требуется исходная траектория и/или код формирования разметки с исходным справочником. Доступные данные не позволяют объявить 247 положительных меток ошибочными либо гарантированно правильными.

По прямому решению пользователя LSGG остается в эксперименте с явной оговоркой о качестве исходной разметки. Массовая замена ILS, слияние попыток и исключение групп не применяются. При моделировании отдельно проверяется чувствительность результатов к участию LSGG и групп обозначений. Табличное решение по LSGG зафиксировано; независимая траекторная валидация не объявляется выполненной.

## Артефакты и воспроизведение

`06_02_airport_audit.json` – контрольный журнал; CSV с префиксом `06_02_airport_` – распределения, последовательности, интервалы, расхождения и записи для проверки. Ключи строк сохранены. Текущий справочник и его манифест находятся в `docs/sources/files/lsgg_label_provenance`. `06_02_current_reference_geometry.json` содержит сферическую оценку текущей геометрии.

Из корня проекта: `.\.venv\Scripts\python.exe europe/scripts/audit_lsgg_airport.py`. Скрипт воспроизводит табличные счетчики; геометрическая оценка и текстовый итог сохранены отдельно. Вход и прежние отчеты сохранены. Исторический отчет 06_03 описывает копию до исключения одной записи.

## Обязательная проверка чувствительности при моделировании

Проверка состава уже выполнена, проверка моделей пока не выполнена. [План](06_02_sensitivity_plan.md) фиксирует варианты выборки и сравнение на одинаковых тестовых наблюдениях. Она не заменяет валидацию траекторий.
